In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
import time
import warnings
import os
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. LOAD DATASET
# ─────────────────────────────────────────────
print("=" * 60)
print("  TEXT CLASSIFICATION: Naïve Bayes vs Logistic Regression")
print("=" * 60)

# Use 4 categories for a clear multi-class comparison
categories = [
    'rec.sport.hockey',
    'sci.space',
    'talk.politics.guns',
    'comp.graphics'
]

print("\n[1] Loading 20 Newsgroups dataset...")
train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

print(f"    Train samples : {len(train_data.data)}")
print(f"    Test  samples : {len(test_data.data)}")
print(f"    Categories    : {categories}")

# Quick peek
print("\n[Sample text]")
print(train_data.data[0][:300], "...")

# ─────────────────────────────────────────────
# 2. CLASS DISTRIBUTION
# ─────────────────────────────────────────────
print("\n[2] Class distribution (train):")
dist = pd.Series(train_data.target).map(dict(enumerate(train_data.target_names)))
print(dist.value_counts().to_string())

# ─────────────────────────────────────────────
# 3. BUILD PIPELINES
# ─────────────────────────────────────────────
print("\n[3] Building pipelines...")

pipelines = {
    "Naïve Bayes (Multinomial)": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ('clf',   MultinomialNB(alpha=0.1))
    ]),
    "Naïve Bayes (Complement)": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ('clf',   ComplementNB(alpha=0.1))
    ]),
    "Logistic Regression": Pipeline([
        ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs',
                                     multi_class='multinomial'))
    ]),
}

# ─────────────────────────────────────────────
# 4. TRAIN & EVALUATE
# ─────────────────────────────────────────────
results = {}

for name, pipe in pipelines.items():
    print(f"\n  ── {name} ──")
    t0 = time.time()
    pipe.fit(train_data.data, train_data.target)
    train_time = time.time() - t0

    t0 = time.time()
    preds = pipe.predict(test_data.data)
    pred_time = time.time() - t0

    acc  = accuracy_score(test_data.target, preds)
    f1   = f1_score(test_data.target, preds, average='weighted')

    # Cross-validation on training set
    cv_scores = cross_val_score(pipe, train_data.data, train_data.target,
                                cv=5, scoring='accuracy')

    results[name] = {
        'accuracy'   : acc,
        'f1_weighted': f1,
        'cv_mean'    : cv_scores.mean(),
        'cv_std'     : cv_scores.std(),
        'train_time' : train_time,
        'pred_time'  : pred_time,
        'preds'      : preds,
        'pipe'       : pipe,
    }

    print(f"    Accuracy       : {acc:.4f}")
    print(f"    F1 (weighted)  : {f1:.4f}")
    print(f"    CV Accuracy    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"    Train time     : {train_time:.3f}s  |  Predict time: {pred_time:.4f}s")
    print()
    print(classification_report(
        test_data.target, preds,
        target_names=[c.split('.')[-1] for c in categories]
    ))

# ─────────────────────────────────────────────
# 5. VISUALISATIONS
# ─────────────────────────────────────────────
print("\n[5] Generating comparison plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Naïve Bayes vs Logistic Regression — Text Classification",
             fontsize=16, fontweight='bold', y=1.01)

palette = ['#4C72B0', '#55A868', '#DD8452']
short_names = ['NB-Multinomial', 'NB-Complement', 'Logistic Reg.']
names_list  = list(results.keys())

# ── Plot 1: Accuracy & F1 bar chart ──
ax = axes[0, 0]
x  = np.arange(3)
w  = 0.35
acc_vals = [results[n]['accuracy']    for n in names_list]
f1_vals  = [results[n]['f1_weighted'] for n in names_list]
bars1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy',   color=palette[0], alpha=0.85)
bars2 = ax.bar(x + w/2, f1_vals,  w, label='F1 Weighted', color=palette[1], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(short_names, fontsize=9)
ax.set_ylim(0.85, 1.0); ax.set_title("Accuracy & F1 Score")
ax.legend(); ax.set_ylabel("Score")
for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# ── Plot 2: Cross-validation ──
ax = axes[0, 1]
cv_means = [results[n]['cv_mean'] for n in names_list]
cv_stds  = [results[n]['cv_std']  for n in names_list]
bars = ax.bar(short_names, cv_means, color=palette, alpha=0.85, yerr=cv_stds,
              capsize=5, error_kw={'elinewidth': 1.5})
ax.set_ylim(0.85, 1.0); ax.set_title("5-Fold Cross-Validation Accuracy")
ax.set_ylabel("CV Accuracy"); ax.tick_params(axis='x', labelsize=9)
for bar, m in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{m:.3f}', ha='center', va='bottom', fontsize=8)

# ── Plot 3: Training time ──
ax = axes[0, 2]
train_times = [results[n]['train_time'] for n in names_list]
bars = ax.bar(short_names, train_times, color=palette, alpha=0.85)
ax.set_title("Training Time (seconds)"); ax.set_ylabel("Time (s)")
ax.tick_params(axis='x', labelsize=9)
for bar, t in zip(bars, train_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{t:.3f}s', ha='center', va='bottom', fontsize=9)

# ── Plots 4-6: Confusion matrices ──
short_labels = [c.split('.')[-1] for c in categories]
for idx, name in enumerate(names_list):
    ax = axes[1, idx]
    cm = confusion_matrix(test_data.target, results[name]['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=short_labels, yticklabels=short_labels,
                linewidths=0.5, cbar=False)
    ax.set_title(f"Confusion Matrix\n{short_names[idx]}", fontsize=10)
    ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()

# Create the directory if it doesn't exist
os.makedirs('/mnt/user-data/outputs/', exist_ok=True)

plt.savefig('/mnt/user-data/outputs/nb_vs_lr_comparison.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("    Saved → nb_vs_lr_comparison.png")

# ─────────────────────────────────────────────
# 6. SUMMARY TABLE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    'Model'       : short_names,
    'Accuracy'    : [f"{results[n]['accuracy']:.4f}"    for n in names_list],
    'F1-Weighted' : [f"{results[n]['f1_weighted']:.4f}" for n in names_list],
    'CV Accuracy' : [f"{results[n]['cv_mean']:.4f}±{results[n]['cv_std']:.4f}"
                     for n in names_list],
    'Train (s)'   : [f"{results[n]['train_time']:.3f}"  for n in names_list],
    'Predict (s)' : [f"{results[n]['pred_time']:.4f}"   for n in names_list],
})
print(summary.to_string(index=False))

# ─────────────────────────────────────────────
# 7. TOP FEATURES PER CLASS (LR)
# ─────────────────────────────────────────────
print("\n[7] Top-10 features per class (Logistic Regression):")
lr_pipe  = results["Logistic Regression"]['pipe']
vocab    = lr_pipe.named_steps['tfidf'].get_feature_names_out()
coef     = lr_pipe.named_steps['clf'].coef_

for i, cat in enumerate(categories):
    top10_idx = np.argsort(coef[i])[-10:][::-1]
    top10     = [vocab[j] for j in top10_idx]
    print(f"  {cat.split('.')[-1]:15s}: {', '.join(top10)}")

print("\nDone! ✓")

  TEXT CLASSIFICATION: Naïve Bayes vs Logistic Regression

[1] Loading 20 Newsgroups dataset...
    Train samples : 2323
    Test  samples : 1546
    Categories    : ['rec.sport.hockey', 'sci.space', 'talk.politics.guns', 'comp.graphics']

[Sample text]

Agreed here...I'll never forget Dan Kelly calling the play-by-play in the '87
Canada Cup.  He was masterful!

And Danny Gallivan will _never_ be replaced; even now when I watch HNIC I
remember his voice...when I see an Al MacInnis or Al Iafrate (hey, what's with
these guys named Al who can shoot??) ...

[2] Class distribution (train):
rec.sport.hockey      600
sci.space             593
comp.graphics         584
talk.politics.guns    546

[3] Building pipelines...

  ── Naïve Bayes (Multinomial) ──
    Accuracy       : 0.8887
    F1 (weighted)  : 0.8883
    CV Accuracy    : 0.8997 ± 0.0061
    Train time     : 3.235s  |  Predict time: 1.0279s

              precision    recall  f1-score   support

      hockey       0.92      0.89      